# HW5 - UFO Sightings Visualization
## IS 445 - Data Visualization

Using the NUFORC UFO sightings dataset to create two Altair visualizations for my Jekyll page.

In [9]:
import pandas as pd
import altair as alt

# need this because the dataset has 80k+ rows and altair complains otherwise
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [10]:
# read in the ufo sightings data from the course repo
# this csv has no header row so i gotta name the columns myself
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

col_names = ["datetime", "city", "state", "country", "shape",
             "duration_sec", "duration_text", "comments",
             "date_posted", "latitude", "longitude"]

df = pd.read_csv(url, header=None, names=col_names, low_memory=False)

print("rows:", len(df))
df.head()

rows: 80332


,datetime,city,state,country,shape,duration_sec,duration_text,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


## Data Cleaning
Need to handle missing values and parse the datetime so I can get years out of it.

In [11]:
# drop rows with no country - can't really use those
df = df.dropna(subset=["country"])

# fill in missing shapes as 'unknown'
df["shape"] = df["shape"].fillna("unknown")

# parse datetime so i can pull out year later
# the format is like "10/10/1949 20:30"
df["datetime"] = pd.to_datetime(df["datetime"], format="mixed", errors="coerce")
df["year"] = df["datetime"].dt.year

# get rid of rows where the date didn't parse
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

# clean up the text fields a bit
df["country"] = df["country"].str.strip().str.upper()
df["shape"] = df["shape"].str.strip().str.lower()

print("cleaned rows:", len(df))
print()
print("countries and their counts:")
print(df["country"].value_counts())

cleaned rows: 70662

countries and their counts:
country
US    65114
CA     3000
GB     1905
AU      538
DE      105
Name: count, dtype: int64


In [12]:
# lets see what shapes are most common
print("top 10 ufo shapes reported:")
print(df["shape"].value_counts().head(10))

top 10 ufo shapes reported:
shape
light        14628
triangle      7038
circle        6717
unknown       6560
fireball      5526
other         4889
sphere        4732
disk          4477
oval          3285
formation     2171
Name: count, dtype: int64


## Plot 1: UFO Sightings by Shape (Interactive - Country Dropdown)
This chart shows the count of sightings for the top 10 most reported UFO shapes. 
The dropdown lets you filter by country so you can compare what shapes are reported more in different places.

In [13]:
# grab only the top 10 shapes so the chart isnt a mess
top_shapes = df["shape"].value_counts().head(10).index.tolist()
df_shapes = df[df["shape"].isin(top_shapes)].copy()

# aggregate - count sightings per shape per country
shape_counts = df_shapes.groupby(["shape", "country"]).size().reset_index(name="count")

# set up the dropdown for country filtering
# "All" option shows everything at once
countries_list = ["All"] + sorted(shape_counts["country"].unique().tolist())

dropdown = alt.binding_select(options=countries_list, name="Country: ")
selection = alt.selection_point(fields=["country"], bind=dropdown, value="All")

# build the bar chart
# using tableau10 color scheme bc it has enough distinct colors for the countries
chart1 = alt.Chart(shape_counts).mark_bar().encode(
    x=alt.X("shape:N", sort="-y", title="UFO Shape"),
    y=alt.Y("count:Q", title="Number of Sightings"),
    color=alt.Color("country:N", title="Country",
                    scale=alt.Scale(scheme="tableau10")),
    tooltip=["shape:N", "country:N", "count:Q"]
).add_params(
    selection
).transform_filter(
    (alt.datum.country == selection.country) | (selection.country == "All")
).properties(
    title="UFO Sightings by Shape (Filter by Country)",
    width=550,
    height=400
)

chart1

alt.Chart(...)

In [14]:
# export as json so i can embed it in my jekyll page
chart1.save("plot1.json")
print("saved plot1.json!")

saved plot1.json!


## Plot 2: UFO Sightings Over Time by Shape
Shows how sighting frequency has changed over the decades, broken down by the top 5 most common shapes.

In [15]:
# filter to reasonable year range
df_time = df[(df["year"] >= 1940) & (df["year"] <= 2014)].copy()

# only keep top 5 shapes so the lines are readable
top5 = df_time["shape"].value_counts().head(5).index.tolist()
df_time = df_time[df_time["shape"].isin(top5)]

# count sightings per year per shape
yearly = df_time.groupby(["year", "shape"]).size().reset_index(name="count")

# line chart with points at each year
# using dark2 colormap - its good for categorical data on a white background
chart2 = alt.Chart(yearly).mark_line(point=True).encode(
    x=alt.X("year:O", title="Year",
            axis=alt.Axis(labelAngle=-45, values=list(range(1940, 2015, 5)))),
    y=alt.Y("count:Q", title="Number of Sightings"),
    color=alt.Color("shape:N", title="Shape",
                    scale=alt.Scale(scheme="dark2")),
    tooltip=["year:O", "shape:N", "count:Q"]
).properties(
    title="UFO Sightings Over Time by Shape (Top 5)",
    width=600,
    height=400
)

chart2

alt.Chart(...)

In [16]:
# save this one too
chart2.save("plot2.json")
print("saved plot2.json!")

saved plot2.json!
